# 3D ResNet DL Feature Extraction for ML

Extract final avgpool features from trained `resnet18_3D` models, apply PCA, MinMax-normalize PCA features, and save a ML-ready Excel table.

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

sys.path.append('/host/d/Github/')

import Osteosarcoma.Build_lists.Build_list as Build_list
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Image_3D.Generator_ResNet as Generator
import Osteosarcoma.Image_3D.resnet.model as model_module

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: cuda


In [2]:
# ============================================================
# 2. Define dataset: use random0 split, folds 0-5
# ============================================================

label = 'Prognosis'
label_col = label + '_label'
random_state = 0

patient_list_file = (
    '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/'
    f'image_label_info_set12_5fold_{label.lower()}_random{random_state}.xlsx'
)

data_root = '/host/e/D/Data/Habitats/Jishuitan/resampled_data_new'
out_root = '/host/d/projects/Habitats/radiomics/dl_3d_ml'
feature_np_out_dir = os.path.join(out_root, 'features_numpy')

ff.make_folder([out_root, feature_np_out_dir])

print('patient_list_file:', patient_list_file)
print('data_root:', data_root)
print('out_root:', out_root)

build = Build_list.Build(patient_list_file)


def build_case_df(batch_list):
    fold_list, patient_set_list, patient_index_list, label_list, _, _ = build.__build__(
        batch_list=batch_list,
        label_column_name=label_col,
    )

    rows = []
    for i in range(len(patient_index_list)):
        patient_set = str(patient_set_list[i])
        patient_index = str(patient_index_list[i])
        image_path = os.path.join(data_root, patient_set, patient_index, 'img.nii.gz')
        mask_path = os.path.join(data_root, patient_set, patient_index, 'label.nii.gz')
        bbox_path = os.path.join(data_root, patient_set, patient_index, 'bbox_mask.nii.gz')
        rows.append({
            'Patient_set': patient_set,
            'Patient_index': patient_index,
            'fold': int(fold_list[i]),
            'Label': int(label_list[i]),
            'Image_filepath': image_path,
            'Mask_filepath': mask_path,
            'BBox_filepath': bbox_path,
        })
    return pd.DataFrame(rows)

fold_case_dfs = {fold: build_case_df([fold]) for fold in [0, 1, 2, 3, 4, 5]}
all_case_df = pd.concat([fold_case_dfs[fold] for fold in [0, 1, 2, 3, 4, 5]], ignore_index=True)

print('All cases:', all_case_df.shape)
print(all_case_df.groupby('fold')['Label'].agg(['count', 'mean']))
all_case_df.head()


patient_list_file: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_random0.xlsx
data_root: /host/e/D/Data/Habitats/Jishuitan/resampled_data_new
out_root: /host/d/projects/Habitats/radiomics/dl_3d_ml
All cases: (330, 7)
      count      mean
fold                 
0        47  0.297872
1        47  0.297872
2        46  0.282609
3        46  0.304348
4        46  0.304348
5        98  0.295918


,Patient_set,Patient_index,fold,Label,Image_filepath,Mask_filepath,BBox_filepath
0,set_1,5,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
1,set_1,18,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
2,set_1,19,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
3,set_1,30,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
4,set_1,34,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...


In [3]:
# ============================================================
# 3. Manually define which model is used for each fold
# ============================================================
# Fill these paths before running feature extraction.
# If you use one all-data 3D model for all cases, put the same checkpoint path
# for fold 0-4. Fold 5 can later use one selected model or the average of five.

model_depth = 18
in_channels = 3
image_size = (96, 96, 64)
batch_size = 4

fold_model_paths = {
    0: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt',
    1: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt',
    2: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt',
    3: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt',
    4: '/host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt',
}

for fold, path in fold_model_paths.items():
    print(f'fold{fold}_model_path:', path)


fold0_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt
fold1_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt
fold2_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt
fold3_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt
fold4_model_path: /host/d/projects/Habitats/models/Prognosis/resnet18_3D_FTall_AUGfull_96x96x64_nomed_adam/random0_all_fold5/models/model-10.pt


In [4]:
# ============================================================
# 4. Helper functions: model loading and 3D avgpool feature extraction
# ============================================================

def load_resnet_checkpoint(model, checkpoint_path):
    if checkpoint_path is None or str(checkpoint_path).strip() == '':
        raise ValueError('checkpoint_path is empty. Please fill fold_model_paths first.')
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(checkpoint_path)

    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint

    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith('module.'):
            key = key[len('module.'):]
        cleaned[key] = value

    model.load_state_dict(cleaned, strict=True)
    return model


def build_loaded_model(checkpoint_path):
    model = model_module.build_resnet3d_model(
        model_depth=model_depth,
        num_classes=2,
        in_channels=in_channels,
    )
    model = load_resnet_checkpoint(model, checkpoint_path)
    model.to(DEVICE)
    model.eval()
    return model


def resnet3d_avgpool_features(model, x):
    """Return final adaptive-avgpool vector from the 3D ResNet backbone."""
    return model.forward_features(x)


def make_dataset(case_df):
    return Generator.Dataset_3D(
        patient_set_list=case_df['Patient_set'].astype(str).tolist(),
        patient_index_list=case_df['Patient_index'].astype(str).tolist(),
        x_file_list=case_df['Image_filepath'].astype(str).tolist(),
        y_list=case_df['Label'].astype(int).tolist(),
        data_root=data_root,
        target_image_size=image_size,
        normalize_factor='medicalnet',
        only_tumor_pixels='seg',
        augment_context='simple',
        shuffle=False,
        augment=False,
        augment_frequency=0,
    )


def extract_features_for_cases(model, case_df):
    dataset = make_dataset(case_df)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    features = []
    labels = []

    model.eval()
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            feat = resnet3d_avgpool_features(model, batch_x)
            features.append(feat.detach().cpu().numpy())
            labels.extend(batch_y.cpu().numpy().astype(int).tolist())

    features = np.concatenate(features, axis=0) if len(features) > 0 else np.zeros((0, 0), dtype=np.float32)
    labels = np.asarray(labels).astype(int)

    if len(case_df) != features.shape[0]:
        raise RuntimeError(f'Feature number mismatch: cases={len(case_df)}, features={features.shape[0]}')

    return features.astype(np.float32), labels


In [8]:
# ============================================================
# 5. Extract DL features for folds 0-4 using their corresponding models
# ============================================================

fold01234_feature_list = []
fold01234_meta_list = []

for fold in [0, 1, 2, 3, 4]:
    print('\n============================================================')
    print('Extracting fold:', fold)
    case_df = fold_case_dfs[fold].copy().reset_index(drop=True)
    model = build_loaded_model(fold_model_paths[fold])
    features, labels_check = extract_features_for_cases(model, case_df)

    if not np.array_equal(labels_check, case_df['Label'].astype(int).to_numpy()):
        raise RuntimeError(f'Label mismatch during feature extraction for fold {fold}.')

    fold01234_feature_list.append(features)
    fold01234_meta_list.append(case_df)

    np.save(os.path.join(feature_np_out_dir, f'fold{fold}_DLfeature.npy'), features)
    case_df.to_excel(os.path.join(feature_np_out_dir, f'fold{fold}_metadata.xlsx'), index=False)

fold01234_DLfeature = np.concatenate(fold01234_feature_list, axis=0)
fold01234_metadata = pd.concat(fold01234_meta_list, ignore_index=True)

np.save(os.path.join(feature_np_out_dir, 'fold01234_DLfeature.npy'), fold01234_DLfeature)
fold01234_metadata.to_excel(os.path.join(feature_np_out_dir, 'fold01234_metadata.xlsx'), index=False)

print('fold01234 feature shape:', fold01234_DLfeature.shape)
fold01234_metadata.head()



Extracting fold: 0

Extracting fold: 1

Extracting fold: 2

Extracting fold: 3

Extracting fold: 4
fold01234 feature shape: (232, 512)


,Patient_set,Patient_index,fold,Label,Image_filepath,Mask_filepath,BBox_filepath
0,set_1,5,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
1,set_1,18,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
2,set_1,19,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
3,set_1,30,0,0,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...
4,set_1,34,0,1,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...


In [6]:
# ============================================================
# 6. Extract DL features for fold 5 using all five fold models
# ============================================================

fold5_df = fold_case_dfs[5].copy().reset_index(drop=True)
fold5_features_by_model = {}

for model_fold in [0,1,2,3,4]:# [0, 1, 2, 3, 4]:
    print('\n============================================================')
    print('Extracting fold5 with model from fold:', model_fold)
    model = build_loaded_model(fold_model_paths[model_fold])
    features, labels_check = extract_features_for_cases(model, fold5_df)

    if not np.array_equal(labels_check, fold5_df['Label'].astype(int).to_numpy()):
        raise RuntimeError(f'Label mismatch during fold5 extraction with model {model_fold}.')

    fold5_features_by_model[model_fold] = features
    np.save(os.path.join(feature_np_out_dir, f'fold5_DLfeature_model{model_fold}.npy'), features)

fold5_stack = np.stack([fold5_features_by_model[k] for k in [0, 1, 2, 3, 4]], axis=0)
fold5_DLfeature_avg = np.mean(fold5_stack, axis=0).astype(np.float32)

np.save(os.path.join(feature_np_out_dir, 'fold5_DLfeature_avg.npy'), fold5_DLfeature_avg)
fold5_df.to_excel(os.path.join(feature_np_out_dir, 'fold5_metadata.xlsx'), index=False)

print('fold5 one-model feature shape:', fold5_features_by_model[0].shape)
print('fold5 averaged feature shape:', fold5_DLfeature_avg.shape)



Extracting fold5 with model from fold: 0

Extracting fold5 with model from fold: 1

Extracting fold5 with model from fold: 2

Extracting fold5 with model from fold: 3

Extracting fold5 with model from fold: 4
fold5 one-model feature shape: (98, 512)
fold5 averaged feature shape: (98, 512)


In [9]:
# ============================================================
# 7. Manually choose which fold5 feature set to use
# ============================================================
# Options:
#   'avg' -> average of five model features
#   0,1,2,3,4 -> features extracted by one specific fold model

fold5_feature_choice = 0

if fold5_feature_choice == 'avg':
    fold5_DLfeature_selected = fold5_DLfeature_avg
elif fold5_feature_choice in [0, 1, 2, 3, 4]:
    fold5_DLfeature_selected = fold5_features_by_model[int(fold5_feature_choice)]
else:
    raise ValueError("fold5_feature_choice must be 'avg' or one of 0,1,2,3,4")

print('Selected fold5 feature:', fold5_feature_choice)
print('Selected fold5 feature shape:', fold5_DLfeature_selected.shape)


Selected fold5 feature: 0
Selected fold5 feature shape: (98, 512)


In [11]:
# ============================================================
# 8. PCA on raw DL features, then MinMax normalize PCA features
# ============================================================

all_DLfeature_raw = np.concatenate(
    [
        fold01234_DLfeature,
        fold5_DLfeature_selected,
    ],
    axis=0,
)

all_metadata = pd.concat(
    [
        fold01234_metadata,
        fold5_df,
    ],
    ignore_index=True,
)

print('All raw DL feature shape:', all_DLfeature_raw.shape)
print('All metadata shape:', all_metadata.shape)

pca_n_components = 0.99

pca = PCA(
    n_components=pca_n_components,
    random_state=random_state,
)

all_DLfeature_pca_raw = pca.fit_transform(all_DLfeature_raw)

print('Raw PCA feature shape:', all_DLfeature_pca_raw.shape)
print('Explained variance ratio sum:', float(np.sum(pca.explained_variance_ratio_)))
print('Raw PCA min:', float(np.min(all_DLfeature_pca_raw)))
print('Raw PCA max:', float(np.max(all_DLfeature_pca_raw)))

pca_minmax_scaler = MinMaxScaler(feature_range=(0, 1))
all_DLfeature_pca = pca_minmax_scaler.fit_transform(all_DLfeature_pca_raw)

print('Final normalized PCA feature shape:', all_DLfeature_pca.shape)
print('Final PCA feature min:', float(np.min(all_DLfeature_pca)))
print('Final PCA feature max:', float(np.max(all_DLfeature_pca)))

os.makedirs(feature_np_out_dir, exist_ok=True)

np.save(os.path.join(feature_np_out_dir, 'all_DLfeature_raw_selected.npy'), all_DLfeature_raw)
np.save(os.path.join(feature_np_out_dir, 'all_DLfeature_PCA_raw.npy'), all_DLfeature_pca_raw)
np.save(os.path.join(feature_np_out_dir, 'all_DLfeature_PCA_minmax.npy'), all_DLfeature_pca)

joblib.dump(pca, os.path.join(feature_np_out_dir, 'pca_model.joblib'))
joblib.dump(pca_minmax_scaler, os.path.join(feature_np_out_dir, 'pca_minmax_scaler.joblib'))

pca_info = {
    'pca_n_components': pca_n_components,
    'n_components_selected': int(all_DLfeature_pca.shape[1]),
    'explained_variance_ratio_sum': float(np.sum(pca.explained_variance_ratio_)),
    'raw_feature_shape': list(all_DLfeature_raw.shape),
    'raw_pca_feature_shape': list(all_DLfeature_pca_raw.shape),
    'final_minmax_pca_feature_shape': list(all_DLfeature_pca.shape),
    'fold5_feature_choice': str(fold5_feature_choice),
}

with open(os.path.join(feature_np_out_dir, 'pca_info.json'), 'w') as f:
    json.dump(pca_info, f, indent=4)

print('Saved PCA artifacts to:', feature_np_out_dir)


All raw DL feature shape: (330, 512)
All metadata shape: (330, 7)
Raw PCA feature shape: (330, 31)
Explained variance ratio sum: 0.9902433156967163
Raw PCA min: -10.383594512939453
Raw PCA max: 14.058733940124512
Final normalized PCA feature shape: (330, 31)
Final PCA feature min: 0.0
Final PCA feature max: 1.0
Saved PCA artifacts to: /host/d/projects/Habitats/radiomics/dl_3d_ml/features_numpy


In [12]:
# ============================================================
# 9. Save final MinMax-normalized PCA DL features as Excel table
# ============================================================

feature_columns = [
    f'DL_feature_{i+1:03d}'
    for i in range(all_DLfeature_pca.shape[1])
]

feature_df = pd.DataFrame(
    all_DLfeature_pca,
    columns=feature_columns,
)

metadata_columns = [
    'Patient_set',
    'Patient_index',
    'Image_filepath',
    'Mask_filepath',
    'fold',
    'Label',
]

ml_table_df = pd.concat(
    [
        all_metadata[metadata_columns].reset_index(drop=True),
        feature_df.reset_index(drop=True),
    ],
    axis=1,
)

save_path = os.path.join(
    out_root,
    'dl_3d_features_PCA.xlsx',
)

ml_table_df.to_excel(save_path, index=False)

print('Saved final MinMax-normalized PCA 3D DL feature table:')
print(save_path)
print('Shape:', ml_table_df.shape)

feature_cols_check = [
    col for col in ml_table_df.columns
    if col.startswith('DL_feature_')
]

print('Final feature min:', float(ml_table_df[feature_cols_check].min().min()))
print('Final feature max:', float(ml_table_df[feature_cols_check].max().max()))

ml_table_df.head()


Saved final MinMax-normalized PCA 3D DL feature table:
/host/d/projects/Habitats/radiomics/dl_3d_ml/dl_3d_features_PCA.xlsx
Shape: (330, 37)
Final feature min: 0.0
Final feature max: 1.0


,Patient_set,Patient_index,Image_filepath,Mask_filepath,fold,Label,DL_feature_001,DL_feature_002,DL_feature_003,DL_feature_004,...,DL_feature_022,DL_feature_023,DL_feature_024,DL_feature_025,DL_feature_026,DL_feature_027,DL_feature_028,DL_feature_029,DL_feature_030,DL_feature_031
0,set_1,5,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,1,0.114760,0.309598,0.293519,0.482003,...,0.063392,0.437120,0.434828,0.357831,0.614724,0.632397,0.397880,0.176818,0.309895,0.373990
1,set_1,18,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,0,0.686792,0.370435,0.204311,0.315277,...,0.575217,0.505287,0.454363,0.498105,0.491545,0.559659,0.304995,0.390536,0.368946,0.345198
2,set_1,19,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,1,0.203086,0.202728,0.161592,0.267012,...,0.566978,0.460461,0.340548,0.412557,0.731759,0.502838,0.390695,0.289732,0.324827,0.437891
3,set_1,30,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,0,0.153087,0.366406,0.525789,0.830651,...,0.351399,0.419995,0.448656,0.608828,0.504918,0.478573,0.194470,0.319681,0.450541,0.388514
4,set_1,34,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,/host/e/D/Data/Habitats/Jishuitan/resampled_da...,0,1,0.265845,0.350735,0.208659,0.342241,...,0.483404,0.529935,0.314938,0.333845,0.655355,0.236469,0.429287,0.402643,0.588770,0.504476
